# Transfer Learning con Keras 3

Entrenar redes neuronales convolucionales (CNNs) puede tomar mucho tiempo y requiere grandes cantidades de datos. Sin embargo, gran parte de ese tiempo se dedica a aprender filtros de bajo nivel que una red utiliza para extraer patrones de las imágenes.

Surge entonces una pregunta natural: **¿podemos usar una red neuronal entrenada en un conjunto de datos y adaptarla para clasificar imágenes diferentes sin entrenar todo desde cero?**

A este enfoque se le llama **aprendizaje por transferencia (transfer learning)**. Normalmente partimos de un modelo preentrenado sobre un conjunto grande, como ImageNet. Ese modelo ya ha aprendido características visuales generales —bordes, texturas, formas y patrones— y podemos reutilizarlas para una nueva tarea.

> En este notebook usaremos **Keras 3** con backend TensorFlow.

In [ ]:
# IMPORTANTE: definir el backend antes de importar keras.
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers
import tensorflow as tf

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
import zipfile
import glob

print("Keras:", keras.__version__)
print("TensorFlow:", tf.__version__)
print("GPU disponibles:", tf.config.list_physical_devices("GPU"))

## Conjunto de datos de Gatos vs. Perros

Resolveremos un problema de clasificación binaria: distinguir imágenes de **gatos** y **perros**.

Usaremos el conjunto de datos *Cats vs. Dogs*, originalmente publicado en Kaggle y distribuido también por Microsoft. Lo descargaremos en el directorio `data/`.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"

zip_path = keras.utils.get_file(
    fname="kagglecatsanddogs_5340.zip",
    origin=url,
    cache_dir=".",
    cache_subdir="data",
)

zip_path = Path(zip_path)
print("Archivo:", zip_path)

In [ ]:
extract_dir = Path("data")
pet_dir = extract_dir / "PetImages"

if not pet_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

print("Dataset:", pet_dir)

### Limpieza de imágenes corruptas

Este conjunto contiene algunas imágenes corruptas. Antes de construir el dataset verificaremos los archivos y eliminaremos únicamente aquellos que no puedan abrirse correctamente.

En PyTorch el notebook original delegaba esta tarea a una utilidad externa. Aquí la implementamos directamente con Pillow para que el notebook no dependa de `pytorchcv`.

In [ ]:
def remove_invalid_images(root_dir):
    root_dir = Path(root_dir)
    bad_files = []

    for path in root_dir.rglob("*"):
        if path.suffix.lower() not in [
            ".jpg", ".jpeg", ".png", ".bmp", ".gif"
        ]:
            continue

        try:
            raw = tf.io.read_file(str(path))
            tf.io.decode_image(
                raw,
                channels=3,
                expand_animations=False
            )

        except Exception:
            bad_files.append(path)

    for path in bad_files:
        print("Eliminando:", path)
        path.unlink(missing_ok=True)

    print(f"\nImágenes inválidas eliminadas: {len(bad_files)}")

remove_invalid_images("data/PetImages")

In [ ]:
import tensorflow as tf

root = Path("data/PetImages")

bad_files = []

for path in root.rglob("*"):
    if path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp", ".gif"]:
        continue

    try:
        raw = tf.io.read_file(str(path))
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    except Exception as e:
        print("ERROR:", path)
        print(e)
        print("-" * 80)
        bad_files.append(path)

print("Archivos problemáticos:", len(bad_files))

## Carga de imágenes con Keras 3

En Keras podemos construir directamente un `tf.data.Dataset` desde una estructura de carpetas:

```text
PetImages/
├── Cat/
└── Dog/
```

Usaremos imágenes de `224×224`, que es el tamaño estándar de entrada de VGG16.

No normalizaremos manualmente con media y desviación estándar. Cada arquitectura preentrenada de Keras Applications proporciona su propia función `preprocess_input`, que aplica exactamente el preprocesamiento esperado por los pesos de ImageNet.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = keras.utils.image_dataset_from_directory(
    "data/PetImages",
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

test_ds = keras.utils.image_dataset_from_directory(
    "data/PetImages",
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
)

class_names = train_ds.class_names
print("Clases:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
def display_dataset(dataset, class_names, n=8):
    images, labels = next(iter(dataset))

    plt.figure(figsize=(14, 6))
    for i in range(min(n, len(images))):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
    plt.tight_layout()

display_dataset(train_ds, class_names)

## Modelos preentrenados

Keras incluye múltiples arquitecturas preentrenadas mediante `keras.applications`.

Comenzaremos con **VGG16** y sus pesos entrenados sobre ImageNet.

In [ ]:
vgg_imagenet = keras.applications.VGG16(
    weights="imagenet",
    include_top=True,
)

images, labels = next(iter(train_ds))
sample_image = images[:1]

x = keras.applications.vgg16.preprocess_input(sample_image)
pred = vgg_imagenet(x, training=False)

print("Índice de la clase con mayor probabilidad:", int(tf.argmax(pred[0])))

El resultado anterior es un índice correspondiente a una de las 1000 clases de ImageNet. En Keras no necesitamos descargar manualmente una tabla externa: podemos decodificar las predicciones con `decode_predictions`.

In [ ]:
top3 = keras.applications.vgg16.decode_predictions(pred.numpy(), top=3)[0]

for imagenet_id, label, probability in top3:
    print(f"{label:25s} {probability:.4f}")

### Arquitectura de VGG16

`Model.summary()` muestra directamente las capas, dimensiones y número de parámetros. No necesitamos instalar `torchsummary`.

In [ ]:
vgg_imagenet.summary()

### Cálculos en GPU

Con Keras + TensorFlow **no es necesario mover manualmente el modelo y cada tensor a `cuda`**. TensorFlow selecciona automáticamente la GPU para las operaciones compatibles cuando existe una disponible.

Podemos comprobar los dispositivos detectados:

In [ ]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("Keras/TensorFlow puede usar GPU:", gpus)
else:
    print("No se detectó GPU; el notebook se ejecutará en CPU.")

## Extracción de características con VGG16

Para utilizar VGG16 como **extractor de características**, cargamos la red con `include_top=False`. Esto elimina el clasificador original de ImageNet y conserva únicamente la parte convolucional.

Para una entrada `224×224×3`, VGG16 produce un tensor de características de tamaño `7×7×512`.

In [ ]:
feature_extractor = keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3),
)

sample_preprocessed = keras.applications.vgg16.preprocess_input(sample_image)
features = feature_extractor(sample_preprocessed, training=False)

print("Shape:", features.shape)

In [ ]:
feature_map = features[0].numpy()          # (7, 7, 512)
feature_map_2d = feature_map.reshape(-1, feature_map.shape[-1]).T  # (512, 49)

plt.figure(figsize=(15, 5))
plt.imshow(feature_map_2d, aspect="auto")
plt.xlabel("Posición espacial (7 × 7)")
plt.ylabel("Canal / filtro")
plt.title("Activaciones de VGG16")
plt.colorbar()
plt.show()

La salida tiene 512 mapas de características sobre una rejilla espacial `7×7`. Para visualizarla la reorganizamos como una matriz `512×49`.

Ahora comprobaremos si esas características sirven para separar gatos y perros. Tomaremos **800 imágenes**, calcularemos sus representaciones con VGG16 una sola vez y entrenaremos sobre ellas un clasificador denso muy pequeño.

In [ ]:
NUM_FEATURE_SAMPLES = 800

small_ds = (
    train_ds
    .unbatch()
    .shuffle(5000, seed=SEED)
    .take(NUM_FEATURE_SAMPLES)
    .batch(32)
)

feature_batches = []
label_batches = []

for images, labels in small_ds:
    images = keras.applications.vgg16.preprocess_input(images)
    f = feature_extractor(images, training=False)
    f = tf.reshape(f, [tf.shape(f)[0], -1])

    feature_batches.append(f)
    label_batches.append(labels)

X_features = tf.concat(feature_batches, axis=0)
y_features = tf.concat(label_batches, axis=0)

print("X:", X_features.shape)
print("y:", y_features.shape)

Dividimos esas 800 representaciones en 700 ejemplos para entrenamiento y 100 para prueba, y usamos una única capa `Dense(2)` como clasificador.

In [ ]:
X_train, X_test = X_features[:700], X_features[700:800]
y_train, y_test = y_features[:700], y_features[700:800]

feature_classifier = keras.Sequential([
    keras.Input(shape=(512 * 7 * 7,)),
    layers.Dense(2, activation="softmax"),
], name="feature_classifier")

feature_classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

history_features = feature_classifier.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=32,
)

Este experimento ilustra la idea central: **el modelo preentrenado transforma las imágenes en características útiles y nosotros aprendemos únicamente una nueva frontera de decisión**.

Sin embargo, no es necesario precalcular manualmente las características. Podemos construir un solo modelo que contenga el extractor VGG16 congelado y un clasificador nuevo.

## Transfer learning con VGG16

Construiremos el modelo en tres partes:

1. `preprocess_input` adapta las imágenes al formato esperado por VGG16.
2. `base_model` contiene la parte convolucional preentrenada y permanece congelada.
3. Una nueva capa `Dense(2)` aprende nuestra tarea gatos vs. perros.

Aquí reemplazamos **todo el clasificador original de ImageNet**, no únicamente su última neurona de salida. De este modo quedan aproximadamente 50 mil parámetros entrenables, que corresponde al escenario descrito conceptualmente en el notebook.

In [ ]:
base_model = keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3),
)

base_model.trainable = False

inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
x = keras.applications.vgg16.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.Flatten(name="flatten")(x)
outputs = layers.Dense(2, activation="softmax", name="classifier")(x)

model = keras.Model(inputs, outputs, name="cats_vs_dogs_vgg16")

model.summary()

print(f"Parámetros totales:      {model.count_params():,}")
print(
    "Parámetros entrenables:",
    f"{sum(np.prod(v.shape) for v in model.trainable_weights):,}"
)

Con el extractor congelado, los gradientes solo actualizan la nueva capa de clasificación. Esto permite adaptar el modelo con pocos datos y reduce considerablemente el costo del backward pass.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

history_transfer = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=1,
)

### Guardar el modelo

El formato recomendado de Keras guarda arquitectura, pesos y configuración del modelo en un único archivo `.keras`.

In [ ]:
Path("data").mkdir(exist_ok=True)

model.save("data/cats_dogs.keras")
print("Modelo guardado en data/cats_dogs.keras")

También podemos guardar **solo los pesos**, que es el concepto más cercano a guardar un `state_dict` en PyTorch.

En Keras:

```python
model.save_weights("data/cats_dogs.weights.h5")
```

Para recuperar únicamente esos pesos debemos recrear una arquitectura compatible y llamar `load_weights`.

In [ ]:
model.save_weights("data/cats_dogs.weights.h5")

# Opción 1: cargar el modelo completo.
loaded_model = keras.models.load_model("data/cats_dogs.keras")

# Opción 2: si recreamos exactamente la misma arquitectura:
# model.load_weights("data/cats_dogs.weights.h5")

print("Modelo cargado:", loaded_model.name)

## Ajuste fino (*fine-tuning*)

Hasta ahora entrenamos únicamente el clasificador nuevo. El extractor de VGG16 sigue usando exactamente las características aprendidas con ImageNet.

En *fine-tuning* descongelamos una parte de la red preentrenada y continuamos entrenando con una **tasa de aprendizaje muy pequeña**.

La secuencia recomendada es:

1. congelar la base preentrenada;
2. entrenar primero el clasificador nuevo;
3. descongelar únicamente las capas superiores que queremos adaptar;
4. **recompilar** el modelo;
5. continuar el entrenamiento con un `learning_rate` pequeño.

No conviene empezar directamente con toda la red descongelada porque las actualizaciones iniciales del clasificador aleatorio pueden alterar agresivamente los pesos preentrenados.

In [ ]:
# Descongelamos la base...
base_model.trainable = True

# ...pero dejamos congelados todos los bloques excepto block5.
for layer in base_model.layers:
    layer.trainable = layer.name.startswith("block5")

trainable_base_layers = [layer.name for layer in base_model.layers if layer.trainable]

print("Capas entrenables de VGG16:")
for name in trainable_base_layers:
    print(" -", name)

Después de cambiar la propiedad `trainable`, debemos **recompilar** el modelo para que Keras reconstruya el conjunto de variables que el optimizador actualizará.

Durante fine-tuning usamos un `learning_rate` mucho menor que el empleado para entrenar inicialmente el clasificador.

In [ ]:
model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=1e-5,
        weight_decay=1e-4,
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.summary()

In [ ]:
history_finetune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=1,
)

> **Nota sobre Batch Normalization:** VGG16 no contiene capas Batch Normalization, pero otras arquitecturas sí. En modelos con BatchNorm suele ser importante ejecutar la base con `training=False` durante transfer learning/fine-tuning para no modificar accidentalmente sus estadísticas internas. Por eso construimos el modelo con `base_model(x, training=False)`.

## Otros modelos de visión por computador

Keras Applications incluye otras familias preentrenadas como **ResNet**, **Inception**, **MobileNet**, **EfficientNet**, **DenseNet**, entre otras.

El paquete base de Keras Applications no incluye ResNet18. Para mantener este notebook sin dependencias adicionales, utilizaremos **ResNet50** como ejemplo de la familia ResNet.

In [ ]:
resnet = keras.applications.ResNet50(
    weights=None,
    include_top=True,
    classes=1000,
)

resnet.summary()

Al igual que VGG16, ResNet puede utilizarse como extractor de características con `include_top=False` y luego conectarse a un nuevo clasificador.

Una diferencia importante es que ResNet contiene capas de **Batch Normalization** y conexiones residuales. Para transfer learning se aplica el mismo patrón general:

```python
base_model = keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)
base_model.trainable = False
```

Después se agrega la cabeza de clasificación específica para la nueva tarea.

## Normalización por lotes (Batch Normalization)

Batch Normalization normaliza las activaciones intermedias utilizando estadísticas calculadas durante el entrenamiento y mantiene estadísticas móviles para inferencia. Esto ayuda a estabilizar el proceso de optimización de redes profundas.

En un flujo de transfer learning debemos tratar estas capas con cuidado porque sus estadísticas forman parte del conocimiento aprendido durante el preentrenamiento.

## Conclusión

Usando aprendizaje por transferencia podemos construir rápidamente un clasificador para una tarea personalizada reutilizando representaciones visuales aprendidas previamente.

En Keras 3 el flujo puede resumirse así:

1. cargar un modelo de `keras.applications` con pesos de ImageNet;
2. remover su clasificador con `include_top=False`;
3. congelar el modelo base con `trainable = False`;
4. agregar una nueva cabeza de clasificación;
5. entrenar la cabeza;
6. opcionalmente descongelar los últimos bloques y hacer *fine-tuning* con una tasa de aprendizaje pequeña.

En este ejemplo VGG16 ya ha visto categorías relacionadas con gatos y perros durante su preentrenamiento en ImageNet, por lo que la transferencia es especialmente favorable. En dominios más alejados de ImageNet —por ejemplo imágenes médicas, industriales o satelitales— puede ser necesario adaptar una porción mayor de la red y validar cuidadosamente el beneficio del fine-tuning.